# 04 - Fallacy classifier
Fine-tunes DistilRoBERTa on `tasksource/logical-fallacy`. **Use a GPU runtime** (Colab: Runtime > Change runtime type > T4). Training takes a few minutes.

In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules:  # Colab: clone the repo and install dependencies
    if not os.path.isdir("fact-and-logic"):
        subprocess.run(["git", "clone", "https://github.com/Anishkumar25/fact-and-logic.git"], check=True)
    os.chdir("fact-and-logic")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
from src.fallacy import train

res = train(epochs=4)
res

## Per-class results on the test split
Macro-F1 matters more than accuracy here, because some fallacy types are rare.

In [ ]:
from datasets import load_dataset
from sklearn.metrics import classification_report
from src.fallacy import FallacyDetector, LABEL, TEXT

det = FallacyDetector()
test = load_dataset('tasksource/logical-fallacy')['test']
y = [str(x) for x in test[LABEL]]
p = [det.predict(t, top=1)[0]['label'] for t in test[TEXT]]
print(classification_report(y, p, zero_division=0))

In [ ]:
for t in ['You cannot trust his research, he never went to a good university.',
          'Everyone is buying it, so it must be good.',
          'If we allow this, soon everything will collapse.']:
    print(t, '->', det.predict(t))

## Publish to the Hub (optional, free)
The deployed app loads the model from the `FALLACY_MODEL` environment variable. Log in first with `huggingface_hub.login()`, then uncomment.

In [ ]:
# from huggingface_hub import login; login()
# train(epochs=4, push='<your-username>/fact-and-logic-fallacy')